In [1]:

"""
import importlib
import src.data.preprocessor as pre
import src.features.engineer as eng

# Força recarregar os módulos com as alterações salvas
importlib.reload(pre)
importlib.reload(eng)

from src.data.preprocessor import run_preprocessing_pipeline
from src.features.engineer import run_feature_engineering

# Etapa 3
X_train, X_val, X_test, y_train, y_val, y_test = run_preprocessing_pipeline()

# Etapa 4
X_train, X_val, X_test = run_feature_engineering(X_train, X_val, X_test)
"""


%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data.preprocessor import run_preprocessing_pipeline
from src.features.engineer import run_feature_engineering
from src.models.trainer    import run_training_pipeline

# Etapa 3
X_train, X_val, X_test, y_train, y_val, y_test = run_preprocessing_pipeline()

# Etapa 4
X_train, X_val, X_test = run_feature_engineering(X_train, X_val, X_test)

# Etapa 5
run_training_pipeline(X_train, y_train, X_val, y_val)


INFO:src.data.preprocessor: Starting pre-processing pipeline...

INFO:src.data.preprocessor:Removed Columns: ['Czas', 'Czas2', 'label_name', 'stan']
INFO:src.data.preprocessor:Shape after removal: (153205, 9)
INFO:src.data.preprocessor:Redundat columns removed: ['Temperature - suction line']
INFO:src.data.preprocessor:Shape after removal: (153205, 8)
INFO:src.data.preprocessor:
 Nan found:
INFO:src.data.preprocessor:Flow - output    6846
INFO:src.data.preprocessor:  'Flow - output': 6846 NaN → filled with median (63.1859)
INFO:src.data.preprocessor:
 NaN treated. Total filled: 6846
INFO:src.data.preprocessor:
Outliers clipped (threshold=3.0 std):
INFO:src.data.preprocessor:  Pressure - leak line: 309 adjusted values
INFO:src.data.preprocessor:
Dataset Split:
INFO:src.data.preprocessor:  Train:      107,243 samples (70.0%)
INFO:src.data.preprocessor:  Validation:  22,981  samples (15.0%)
INFO:src.data.preprocessor:  Test:        22,981 samples (15.0%)
INFO:src.data.preprocessor:Trained 

  0%|          | 0/50 [00:00<?, ?it/s]

INFO:src.models.trainer:
🏆 Melhor trial:
INFO:src.models.trainer:  f1_macro: 0.9990
INFO:src.models.trainer:  Params:   {'n_estimators': 588, 'max_depth': 10, 'learning_rate': 0.07323409860286129, 'subsample': 0.7838411618943656, 'colsample_bytree': 0.6993053723775049, 'min_child_weight': 1, 'gamma': 0.005426392585129239, 'reg_alpha': 0.24200626556390956, 'reg_lambda': 0.6664844739220643}
INFO:src.models.trainer:
✅ Tuning concluído — melhores params prontos para treino final
INFO:src.models.trainer:✅ MLflow configurado — experimento: 'pump-failure-prediction'
INFO:src.models.trainer:
🔄 Cross-Validation — XGBoost_Tuned...
INFO:src.models.trainer:
📊 Cross-Validation (5 folds):
INFO:src.models.trainer:  cv_f1_macro_mean: 0.9992
INFO:src.models.trainer:  cv_f1_macro_std: 0.0001
INFO:src.models.trainer:  cv_roc_auc_mean: 1.0000
INFO:src.models.trainer:  cv_roc_auc_std: 0.0000
INFO:src.models.trainer:  cv_accuracy_mean: 0.9992
INFO:src.models.trainer:  cv_accuracy_std: 0.0001


AttributeError: 'NoneType' object has no attribute 'items'

In [4]:

from src.models.trainer import train_model, setup_mlflow
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# Melhores params encontrados pelo Optuna
best_params = {
    "n_estimators":      588,
    "max_depth":         10,
    "learning_rate":     0.07323409860286129,
    "subsample":         0.7838411618943656,
    "colsample_bytree":  0.6993053723775049,
    "min_child_weight":  1,
    "gamma":             0.005426392585129239,
    "reg_alpha":         0.24200626556390956,
    "reg_lambda":        0.6664844739220643,
    "eval_metric":       "mlogloss",
    "random_state":      42,
    "n_jobs":            -1,
}

# Treino XGBoost com melhores params
train_model(
    model=XGBClassifier(**best_params),
    model_name="XGBoost_Tuned",
    params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

# Treino Random Forest como baseline
rf_params = {
    "n_estimators": 200,
    "max_depth":    10,
    "random_state": 42,
    "n_jobs":       -1,
}

train_model(
    model=RandomForestClassifier(**rf_params),
    model_name="RandomForest",
    params=rf_params,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)


INFO:src.models.trainer:✅ MLflow configurado — experimento: 'pump-failure-prediction'
INFO:src.models.trainer:
🔄 Cross-Validation — XGBoost_Tuned...
INFO:src.models.trainer:
📊 Cross-Validation (5 folds):
INFO:src.models.trainer:  cv_f1_macro_mean: 0.9992
INFO:src.models.trainer:  cv_f1_macro_std: 0.0001
INFO:src.models.trainer:  cv_roc_auc_mean: 1.0000
INFO:src.models.trainer:  cv_roc_auc_std: 0.0000
INFO:src.models.trainer:  cv_accuracy_mean: 0.9992
INFO:src.models.trainer:  cv_accuracy_std: 0.0001
INFO:src.models.trainer:
🏋️ Treinando XGBoost_Tuned...
INFO:src.models.trainer:
📊 Métricas — XGBoost_Tuned:
INFO:src.models.trainer:  val_f1_macro: 0.9972
INFO:src.models.trainer:  val_f1_weighted: 0.9980
INFO:src.models.trainer:  val_roc_auc: 1.0000
INFO:src.models.trainer:  val_accuracy: 0.9980
INFO:src.models.trainer:  val_f1_normal: 0.9990
INFO:src.models.trainer:  val_f1_desgaste: 0.9979
INFO:src.models.trainer:  val_f1_falha1: 0.9961
INFO:src.models.trainer:  val_f1_falha2: 0.9958
INF

c:\python311\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'PumpFailurePredictor_XGBoost_Tuned'.
Created version '1' of model 'PumpFailurePredictor_XGBoost_Tuned'.
INFO:src.models.trainer:
✅ Run 'XGBoost_Tuned' registrado no MLflow!
INFO:src.models.trainer:✅ MLflow configurado — experimento: 'pump-failure-prediction'
INFO:src.models.trainer:
🔄 Cross-Validation — RandomForest...
c:\python311\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memor

Successfully registered model 'PumpFailurePredictor_RandomForest'.
Created version '1' of model 'PumpFailurePredictor_RandomForest'.
INFO:src.models.trainer:
✅ Run 'RandomForest' registrado no MLflow!


In [5]:

from pathlib import Path
import os

# Mostra onde o MLflow salvou os runs
mlruns_path = Path("C:/git/pump-failure-prediction/mlruns")
print(f"Existe: {mlruns_path.exists()}")
print(f"\nConteúdo:")
for item in mlruns_path.rglob("*"):
    print(item)


Existe: True

Conteúdo:
C:\git\pump-failure-prediction\mlruns\.trash
C:\git\pump-failure-prediction\mlruns\0
C:\git\pump-failure-prediction\mlruns\290697122907952556
C:\git\pump-failure-prediction\mlruns\models
C:\git\pump-failure-prediction\mlruns\0\meta.yaml
C:\git\pump-failure-prediction\mlruns\290697122907952556\1fe6d75d4fdb4b56a349cd0c319aa18f
C:\git\pump-failure-prediction\mlruns\290697122907952556\29e8eb39a1854651ace6138e755474e3
C:\git\pump-failure-prediction\mlruns\290697122907952556\4e033821a58d468f8e71ec205c0068fd
C:\git\pump-failure-prediction\mlruns\290697122907952556\56203276a2a942a39db2fce4f08ea19e
C:\git\pump-failure-prediction\mlruns\290697122907952556\899b2a676d914a889c3f1a1f244cbf9d
C:\git\pump-failure-prediction\mlruns\290697122907952556\dcc0184c37d74839a28c2b547372ef0a
C:\git\pump-failure-prediction\mlruns\290697122907952556\meta.yaml
C:\git\pump-failure-prediction\mlruns\290697122907952556\models
C:\git\pump-failure-prediction\mlruns\290697122907952556\1fe6d75d4fd

In [6]:

from src.models.trainer import train_model
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

best_params = {
    "n_estimators":      588,
    "max_depth":         10,
    "learning_rate":     0.07323409860286129,
    "subsample":         0.7838411618943656,
    "colsample_bytree":  0.6993053723775049,
    "min_child_weight":  1,
    "gamma":             0.005426392585129239,
    "reg_alpha":         0.24200626556390956,
    "reg_lambda":        0.6664844739220643,
    "eval_metric":       "mlogloss",
    "random_state":      42,
    "n_jobs":            -1,
}

train_model(
    model=XGBClassifier(**best_params),
    model_name="XGBoost_Tuned",
    params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

rf_params = {
    "n_estimators": 200,
    "max_depth":    10,
    "random_state": 42,
    "n_jobs":       -1,
}

train_model(
    model=RandomForestClassifier(**rf_params),
    model_name="RandomForest",
    params=rf_params,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)


2026/04/15 21:09:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/15 21:09:45 INFO mlflow.store.db.utils: Updating database tables
2026/04/15 21:09:47 INFO mlflow.tracking.fluent: Experiment with name 'pump-failure-prediction' does not exist. Creating a new experiment.
INFO:src.models.trainer:✅ MLflow configurado — experimento: 'pump-failure-prediction'
INFO:src.models.trainer:
🔄 Cross-Validation — XGBoost_Tuned...
INFO:src.models.trainer:
📊 Cross-Validation (5 folds):
INFO:src.models.trainer:  cv_f1_macro_mean: 0.9992
INFO:src.models.trainer:  cv_f1_macro_std: 0.0001
INFO:src.models.trainer:  cv_roc_auc_mean: 1.0000
INFO:src.models.trainer:  cv_roc_auc_std: 0.0000
INFO:src.models.trainer:  cv_accuracy_mean: 0.9992
INFO:src.models.trainer:  cv_accuracy_std: 0.0001
INFO:src.models.trainer:
🏋️ Treinando XGBoost_Tuned...
INFO:src.models.trainer:
📊 Métricas — XGBoost_Tuned:
INFO:src.models.trainer:  val_f1_macro: 0.9972
INFO:src.models.trainer:  val_f1_weig

Successfully registered model 'PumpFailurePredictor_XGBoost_Tuned'.
Created version '1' of model 'PumpFailurePredictor_XGBoost_Tuned'.
INFO:src.models.trainer:
✅ Run 'XGBoost_Tuned' registrado no MLflow!
INFO:src.models.trainer:✅ MLflow configurado — experimento: 'pump-failure-prediction'
INFO:src.models.trainer:
🔄 Cross-Validation — RandomForest...
INFO:src.models.trainer:
📊 Cross-Validation (5 folds):
INFO:src.models.trainer:  cv_f1_macro_mean: 0.9428
INFO:src.models.trainer:  cv_f1_macro_std: 0.0005
INFO:src.models.trainer:  cv_roc_auc_mean: 0.9953
INFO:src.models.trainer:  cv_roc_auc_std: 0.0001
INFO:src.models.trainer:  cv_accuracy_mean: 0.9430
INFO:src.models.trainer:  cv_accuracy_std: 0.0005
INFO:src.models.trainer:
🏋️ Treinando RandomForest...
INFO:src.models.trainer:
📊 Métricas — RandomForest:
INFO:src.models.trainer:  val_f1_macro: 0.8978
INFO:src.models.trainer:  val_f1_weighted: 0.9363
INFO:src.models.trainer:  val_roc_auc: 0.9945
INFO:src.models.trainer:  val_accuracy: 0.9

Successfully registered model 'PumpFailurePredictor_RandomForest'.
Created version '1' of model 'PumpFailurePredictor_RandomForest'.
INFO:src.models.trainer:
✅ Run 'RandomForest' registrado no MLflow!
